## Start Spark Session

In [1]:
# What happens: Spark engine starts on your local machine using all CPU cores.
# Check Dashboard: Open http://localhost:4040 — confirms Spark is running.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 12:04:52 WARN Utils: Your hostname, codespaces-91f0b4, resolves to a loopback address: 127.0.0.1; using 10.0.3.47 instead (on interface eth0)
26/02/27 12:04:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 12:04:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read Green Taxi Parquet Files

In [2]:
# What happens: Reads all green taxi parquet files across all years and months.
# No job runs yet — Spark is just planning.
df_green = spark.read.parquet('data/pq/green/*/*')

## Register Green as SQL Table

In [3]:
# What happens: Tells Spark to treat df_green as a SQL table called 'green'
# so we can write SQL queries against it.
df_green.registerTempTable('green')

/workspaces/data-engineering-zoomcamp/06-batch/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


## Run GroupBy SQL on Green Taxi (Hourly Revenue)

In [4]:
# What happens: Lazy operation — Spark builds the query plan but does NOT run yet.
# Groups trips by hour and zone, calculates total revenue and trip count.
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

## Write Green Revenue to Parquet

In [5]:
# What happens: NOW Spark runs. Two stages happen internally:
# Stage 1 — Filter records before 2020, then GroupBy within each partition
# Stage 2 — Reshuffle data so same hour+zone keys end up in same partition, then final aggregation
# Check Dashboard: http://localhost:4040 -> Jobs -> click running job -> Stages
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

## Read Yellow Taxi Parquet Files

In [6]:
# What happens: Same as Step 2 but for yellow taxi data.
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

## Register Yellow as SQL Table

In [7]:
# What happens: Same as Step 3 but for yellow taxi data.
df_yellow.registerTempTable('yellow')

## Run GroupBy SQL on Yellow Taxi (Hourly Revenue)

In [8]:
# What happens: Same as Step 4 but for yellow taxi.
# Note: tpep_ prefix instead of lpep_ for yellow taxi datetime columns.
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

## Write Yellow Revenue to Parquet

In [9]:
# What happens: Spark runs the yellow taxi GroupBy job. Same two stages as Step 5.
# Check Dashboard: http://localhost:4040 -> Jobs -> yellow taxi job running with 3 stages
# (the extra stage is repartition)
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

## Read Back the Saved Revenue Files

In [10]:
# What happens: Instead of recomputing, we load the already saved results.
# This is called materializing — saving intermediate results so we don't
# redo expensive computations.
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

## Rename Columns Before Joining

In [11]:
# What happens: Both tables have columns named amount and number_records.
# Renaming them before joining prevents confusion —
# so we know which column belongs to green and which to yellow.
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

## Join Green and Yellow Revenue (Outer Join)

In [12]:
# What happens: Lazy operation — joins both tables by hour and zone.
# outer join means:
# - If a zone had green trips but no yellow trips -> yellow columns will be null
# - If a zone had yellow trips but no green trips -> green columns will be null
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

## Write Join Result to Parquet

In [13]:
# What happens: NOW Spark runs the join using Sort Merge Join.
# Reshuffles both datasets so records with the same hour+zone key
# end up in the same partition, then merges them.
# Check Dashboard: http://localhost:4040 -> Jobs -> Stages:
# Stage 1: GroupBy for green + reshuffling
# Stage 2: GroupBy for yellow + reshuffling
# Stage 3: Sort Merge Join — combining both into one result
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

## Read Back the Join Result

In [14]:
# What happens: Loads the combined green + yellow revenue table.
# No job runs — just reads metadata. You will see the schema printed.
df_join = spark.read.parquet('data/report/revenue/total')
df_join

DataFrame[hour: timestamp, zone: int, green_amount: double, green_number_records: bigint, yellow_amount: double, yellow_number_records: bigint]

## Read Zones Lookup Table

In [15]:
# What happens: Reads the small zones lookup table (only 265 rows).
# This is a very small file compared to the revenue data.
df_zones = spark.read.parquet('zones/')
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

## Join Revenue with Zones (Broadcast Join)

In [16]:
# What happens: Lazy operation — joins the large revenue table with the tiny zones table.
# Spark automatically detects df_zones is very small and uses Broadcast Join.
# Broadcast Join: Spark sends a full copy of the zones table to every executor.
# Each executor does a fast local lookup — no reshuffling needed.
# Much faster than Sort Merge Join.
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

## Drop Duplicate Columns and Save Final Result

In [17]:
# What happens: Removes the duplicate zone ID columns and saves the final result.
# Check Dashboard: http://localhost:4040 -> Jobs -> only 1 stage
# because Broadcast Join needs no reshuffling.
# Compare this to Step 13 which had 3 stages.
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones', mode='overwrite')

----------------------------------------                                        
Exception occurred during processing of request from ('127.0.0.1', 51484)
Traceback (most recent call last):
  File "/home/codespace/.local/share/uv/python/cpython-3.13.11-linux-x86_64-gnu/lib/python3.13/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/codespace/.local/share/uv/python/cpython-3.13.11-linux-x86_64-gnu/lib/python3.13/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/codespace/.local/share/uv/python/cpython-3.13.11-linux-x86_64-gnu/lib/python3.13/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/codespace/.local/share/uv/python/cpython-3.13